# 📝 Linux Commands
### Exercises & Solutions — 28 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- File & directory operations (1-6)
- Text viewing & searching: grep, head, tail, wc (7-12)
- Pipes, redirection, stream processing (13-17)
- sed/awk text transformation (18-21)
- Permissions & ownership (22-24)
- Process management & system commands (25-28)

**Note:** All commands here execute LIVE against this real Linux container,
so output is genuine command output, not simulated text.


---


### 1. Navigate and Create a Nested Directory Structure

Use `mkdir -p` to create a multi-level directory tree in one command, then verify with `find`.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("mkdir -p project/src/components project/tests project/docs/api", wd)
print(sh("find project -type d | sort", wd))

### 2. Copy Files Recursively and Selectively

Use `cp -r` to copy an entire directory tree, then use `cp` with a glob pattern to copy only `.py` files elsewhere.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("mkdir -p src", wd)
for name in ["a.py", "b.py", "c.txt", "d.md"]:
    sh(f"touch src/{name}", wd)

sh("cp -r src src_backup", wd)
print("Full backup:", sh("ls src_backup", wd))

sh("mkdir python_only && cp src/*.py python_only/", wd)
print("\nPython files only:", sh("ls python_only", wd))

### 3. Move and Rename Files in Bulk

Use a shell loop to rename all `.txt` files in a directory to have a `.bak` extension instead.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
for i in range(4):
    sh(f"echo 'data{i}' > file{i}.txt", wd)

print("Before:", sh("ls *.txt", wd))
sh('for f in *.txt; do mv "$f" "${f%.txt}.bak"; done', wd)
print("After:", sh("ls *.bak", wd))

### 4. Find Files by Multiple Criteria

Use `find` combining `-name`, `-size`, and `-mtime` filters to locate specific files matching several conditions at once.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("mkdir -p logs", wd)
sh("dd if=/dev/zero of=logs/big.log bs=1024 count=200 2>/dev/null", wd)
sh("dd if=/dev/zero of=logs/small.log bs=1024 count=5 2>/dev/null", wd)
sh("touch logs/recent.txt", wd)

print("Files larger than 100KB:")
print(sh("find logs -name '*.log' -size +100k", wd))

print("\nAll .log files (regardless of size):")
print(sh("find logs -name '*.log'", wd))

### 5. Calculate Directory Sizes and Find the Largest

Use `du` to measure sizes of multiple subdirectories, then sort to find which one is consuming the most space.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("mkdir -p dirA dirB dirC", wd)
sh("dd if=/dev/zero of=dirA/f.bin bs=1024 count=500 2>/dev/null", wd)
sh("dd if=/dev/zero of=dirB/f.bin bs=1024 count=100 2>/dev/null", wd)
sh("dd if=/dev/zero of=dirC/f.bin bs=1024 count=900 2>/dev/null", wd)

print(sh("du -sh dirA dirB dirC | sort -rh", wd))

### 6. Create Symbolic Links and Understand Them

Create a symlink with `ln -s`, verify it points to the original, and show what happens when the original file is modified.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("echo 'original content' > real_file.txt", wd)
sh("ln -s real_file.txt link_to_file.txt", wd)

print("ls -la showing the symlink:")
print(sh("ls -la link_to_file.txt", wd))

sh("echo 'updated content' > real_file.txt", wd)
print("\nReading through the symlink (reflects the update):")
print(sh("cat link_to_file.txt", wd))

### 7. head and tail with Custom Line Counts

Use `head -n` and `tail -n` with different counts, plus `tail -n +N` to skip the first N lines instead.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("seq 1 20 > numbers.txt", wd)

print("First 3 lines:", sh("head -n 3 numbers.txt", wd).replace(chr(10), ","))
print("Last 3 lines:", sh("tail -n 3 numbers.txt", wd).replace(chr(10), ","))
print("Everything FROM line 15 onward:", sh("tail -n +15 numbers.txt", wd).replace(chr(10), ","))

### 8. grep with Multiple Files and Context Lines

Use `grep -r` to search across multiple files, and `-A`/`-B` to show lines of context AFTER/BEFORE each match.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("mkdir logs", wd)
log_content = "\n".join([f"line {i}" if i != 5 else "line 5 ERROR occurred here" for i in range(1, 11)])
with open(os.path.join(wd, "logs", "app.log"), "w") as f:
    f.write(log_content)

print("grep -r across directory:")
print(sh("grep -r ERROR logs/", wd))

print("\ngrep with context (2 lines before/after):")
print(sh("grep -A 2 -B 2 ERROR logs/app.log", wd))

### 9. grep with Regex Anchors and Character Classes

Use `grep -E` with `^`, `$`, and character classes to match lines starting/ending with specific patterns.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
lines = ["apple", "banana", "cherry123", "123start", "end99", "no_numbers_here"]
with open(os.path.join(wd, "words.txt"), "w") as f:
    f.write("\n".join(lines))

print("Lines starting with a digit:")
print(sh(r"grep -E '^[0-9]' words.txt", wd))

print("\nLines ending with a digit:")
print(sh(r"grep -E '[0-9]$' words.txt", wd))

print("\nLines with NO digits at all (-v invert):")
print(sh(r"grep -vE '[0-9]' words.txt", wd))

### 10. Count Occurrences and Unique Matches

Use `grep -c` for per-file match counts, and `grep -o | sort | uniq -c` to count occurrences of a pattern WITHIN a file.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
text = "error warning error info error warning info error"
with open(os.path.join(wd, "log.txt"), "w") as f:
    f.write(text)

print("Total lines containing 'error':", sh("grep -c error log.txt", wd))
print("\nCount of EACH word occurrence:")
print(sh("grep -o -E 'error|warning|info' log.txt | sort | uniq -c | sort -rn", wd))

### 11. wc for Line, Word, and Character Counts

Use `wc -l`, `wc -w`, `wc -c` individually and together to analyze a text file's size in different dimensions.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
with open(os.path.join(wd, "essay.txt"), "w") as f:
    f.write("The quick brown fox jumps over the lazy dog.\nThis is a second line of text.\n")

print("Lines:", sh("wc -l essay.txt", wd))
print("Words:", sh("wc -w essay.txt", wd))
print("Characters:", sh("wc -c essay.txt", wd))
print("All three together:", sh("wc essay.txt", wd))

### 12. Combine find + grep to Search Code for a Pattern

Use `find ... -exec grep` (or `find | xargs grep`) to search ALL Python files in a tree for a specific function call.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("mkdir -p src/utils src/models", wd)
with open(os.path.join(wd, "src/utils/helpers.py"), "w") as f:
    f.write("import logging\nlogging.warning('test')\n")
with open(os.path.join(wd, "src/models/user.py"), "w") as f:
    f.write("class User:\n    pass\n")
with open(os.path.join(wd, "src/utils/db.py"), "w") as f:
    f.write("import logging\nlogging.error('db issue')\n")

print("Files using the logging module:")
print(sh("find src -name '*.py' -exec grep -l 'import logging' {} \\;", wd))

### 13. Build a Multi-Stage Pipeline (cat | grep | sort | uniq)

Build a 4-stage pipeline counting unique IP addresses in a simulated access log, sorted by frequency.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
import random
random.seed(1)
ips = [f"192.168.1.{random.randint(1,5)}" for _ in range(50)]
with open(os.path.join(wd, "access.log"), "w") as f:
    f.write("\n".join(f"{ip} GET /page HTTP/1.1" for ip in ips))

result = sh("cat access.log | awk '{print $1}' | sort | uniq -c | sort -rn", wd)
print(result)

### 14. Redirect stdout and stderr Separately

Run a command producing BOTH normal output and an error, redirecting each stream to a DIFFERENT file.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("touch exists.txt", wd)
sh("ls exists.txt missing.txt > out.log 2> err.log", wd)

print("stdout file (out.log):")
print(sh("cat out.log", wd))
print("\nstderr file (err.log):")
print(sh("cat err.log", wd))

### 15. Use tee to Both Display AND Save Output

Use `tee` to simultaneously print a command's output to the terminal AND save it to a file, demonstrating the split-stream pattern.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
result = sh("echo 'important log line' | tee saved.log", wd)
print("Command output (shown immediately):", result)
print("\nSAME content also saved to file:")
print(sh("cat saved.log", wd))

# tee -a appends instead of overwriting
sh("echo 'second line' | tee -a saved.log > /dev/null", wd)
print("\nAfter append:")
print(sh("cat saved.log", wd))

### 16. Chain Commands with && and ||

Use `&&` (run only if previous succeeded) and `||` (run only if previous failed) to build conditional command chains.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
print("Success chain (mkdir succeeds, so echo runs):")
print(sh("mkdir newdir && echo 'directory created successfully'", wd))

print("\nFailure chain (mkdir fails - dir exists - so echo via || runs instead):")
print(sh("mkdir newdir || echo 'directory already existed, skipping'", wd))

print("\nCombined: try primary, fall back, then always cleanup with ;")
print(sh("false || echo 'fallback ran' ; echo 'this always runs regardless'", wd))

### 17. Process Substitution and Command Substitution

Use `$(command)` to capture a command's output directly into a variable or another command's arguments.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("echo 'line1' > f1.txt && echo 'line2' >> f1.txt && echo 'line3' >> f1.txt", wd)

result = sh('echo "This file has $(wc -l < f1.txt) lines"', wd)
print(result)

result2 = sh('LATEST=$(ls -t f1.txt | head -1) && echo "Most recent: $LATEST"', wd)
print(result2)

### 18. sed for Simple Find-and-Replace

Use `sed 's/old/new/'` for first-match replacement vs `s/old/new/g` for ALL matches on a line.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
with open(os.path.join(wd, "text.txt"), "w") as f:
    f.write("cat cat cat dog cat\n")

print("Original:", sh("cat text.txt", wd))
print("First match only:", sh("sed 's/cat/dog/' text.txt", wd))
print("All matches (g flag):", sh("sed 's/cat/dog/g' text.txt", wd))

### 19. sed for Line-Specific Operations

Use `sed -n 'Np'` to print only a specific line, and `sed 'Nd'` to delete a specific line by number.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("seq 1 10 | sed 's/^/line/' > numbered.txt", wd)
print("Full file:")
print(sh("cat numbered.txt", wd))

print("\nOnly line 5:")
print(sh("sed -n '5p' numbered.txt", wd))

print("\nFile WITHOUT line 3 (deleted):")
print(sh("sed '3d' numbered.txt", wd))

### 20. awk for Field Extraction and Conditional Filtering

Use `awk` to extract specific columns from structured text AND apply a numeric filter condition in the same command.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
csv_data = "Alice,Engineering,95000\nBob,Marketing,72000\nCarol,Engineering,105000\nDave,Sales,68000"
with open(os.path.join(wd, "employees.csv"), "w") as f:
    f.write(csv_data)

print("Names and salaries only:")
print(sh("awk -F',' '{print $1, $3}' employees.csv", wd))

print("\nOnly employees earning over 80000:")
print(sh("awk -F',' '$3 > 80000 {print $1, $3}' employees.csv", wd))

### 21. awk for Aggregation (Sum, Average, Count)

Use `awk`'s `BEGIN`/`END` blocks to compute a running sum and final average across all rows of a file.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
with open(os.path.join(wd, "sales.txt"), "w") as f:
    f.write("100\n250\n175\n300\n90\n")

result = sh("""awk 'BEGIN {sum=0; count=0} {sum+=$1; count++} END {print "Total:", sum; print "Count:", count; print "Average:", sum/count}' sales.txt""", wd)
print(result)

### 22. Set and Verify Exact Permissions with chmod

Set a file to EXACTLY `rwxr-x---` (750) using numeric chmod, then verify with `ls -l` and `stat`.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("touch script.sh", wd)
sh("chmod 750 script.sh", wd)
print(sh("ls -l script.sh", wd))
print(sh("stat -c '%A %a %n' script.sh", wd))

### 23. Recursively Fix Permissions on a Directory Tree

Use `chmod -R` with `find`-based filtering to set DIFFERENT permissions for directories (755) vs files (644) in one tree.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("mkdir -p proj/src proj/docs", wd)
sh("touch proj/src/a.py proj/docs/readme.md", wd)
sh("chmod -R 777 proj", wd)   # start messy/overpermissive

print("Before fix:")
print(sh("find proj -printf '%m %p\\n'", wd))

sh("find proj -type d -exec chmod 755 {} \\;", wd)
sh("find proj -type f -exec chmod 644 {} \\;", wd)

print("\nAfter fix (dirs=755, files=644):")
print(sh("find proj -printf '%m %p\\n'", wd))

### 24. Find and Report Security-Risky Permissions

Write a one-liner finding all WORLD-WRITABLE files in a tree — a common security audit task.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("touch safe1.txt safe2.txt", wd)
sh("chmod 644 safe1.txt safe2.txt", wd)
sh("touch risky.txt", wd)
sh("chmod 666 risky.txt", wd)   # world-writable - a real security smell

print("Security audit - world-writable files found:")
print(sh("find . -type f -perm -o+w", wd) or "(none found)")

### 25. List and Filter Running Processes

Use `ps aux` combined with `grep` to find specific running processes by name (a very common real-world debugging task).

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
print("Process list header + python processes (filtering out the grep itself):")
result = sh("ps aux | head -1", wd)
print(result)
result2 = sh("ps aux | grep -i python | grep -v grep | head -3", wd)
print(result2 or "(no matching processes shown in this sandboxed output)")

### 26. Background a Long-Running Process and Check on It

Start a background process with `&`, confirm the shell returns immediately, then use `jobs` to check its status.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
print(sh("sleep 3 & echo \"Shell returned immediately, PID: $!\"", wd))
print("\nThe '&' backgrounds it; '$!' captures the PID of the last backgrounded process")

### 27. Measure Disk Usage and Free Space

Use `df -h` for filesystem-level free space and `du -h --max-depth=1` for a one-level breakdown of a directory's contents.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
sh("mkdir -p a b c", wd)
sh("dd if=/dev/zero of=a/f.bin bs=1024 count=100 2>/dev/null", wd)
sh("dd if=/dev/zero of=b/f.bin bs=1024 count=200 2>/dev/null", wd)
sh("dd if=/dev/zero of=c/f.bin bs=1024 count=50 2>/dev/null", wd)

print("One-level breakdown:")
print(sh("du -h --max-depth=1", wd))

print("\nFilesystem-level free space (df -h, first 2 lines):")
print(sh("df -h | head -2", wd))

### 28. Build a Complete Diagnostic One-Liner (Real Incident-Response Pattern)

Combine grep, awk, sort, and uniq into ONE comprehensive pipeline that summarizes error frequency by hour from a realistic log file.

In [ ]:
import subprocess, os, tempfile

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    return (r.stdout + r.stderr).rstrip()

wd = tempfile.mkdtemp()
import random
random.seed(5)
lines = []
for h in range(24):
    for _ in range(random.randint(0, 8)):
        level = random.choice(["INFO"]*7 + ["ERROR"]*2 + ["WARNING"])
        lines.append(f"2024-01-15 {h:02d}:{random.randint(0,59):02d}:00 {level} sample message")
with open(os.path.join(wd, "service.log"), "w") as f:
    f.write("\n".join(lines))

pipeline = "grep ERROR service.log | awk '{print substr($2,1,2)}' | sort | uniq -c | sort -rn"
print(f"Pipeline: {pipeline}\n")
print("Error count by hour (busiest hours first):")
print(sh(pipeline, wd))